# HW03 Data Science Academy
# Мурзабеков Эльдар 

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.

In [62]:
import numpy as np

Мои предсказания и объяснения:

1. arange создаёт числа от 0 до 5. С помощью reshape делаем из них массив 2*3. По индексу [1, 2] получаем 5, а shape показывает размер массива (2, 3).

In [63]:
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)

5 (2, 3)


2. Создаем массив и с помощью argmax достаем индекс наибольшего числа, с помощью max соответственно наибольшее. Вывод 1 и 8, ибо 8 было 2 раза, но за максимум будет считаться первый попавшийся

In [64]:
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())

1 8


3. arange создает числа до 10, b становится ссылкой на a и берет каждый третий элемент. Затем переназначает индекс 0 и выводит 99

In [65]:
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])

99


4. Создаем массив и с помощью фильтрации чисел больше 3 суммируем, затем выводим среднее. Будет 2, 0.5

In [66]:
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())

2 0.5


5. Массив из нулей с 2 строками и 3 столбцами. Массив с пятью точками от нуля до одного по индексу 1, вывод 0.25

In [67]:
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])

(2, 3) 0.25


In [68]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [69]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

(5463,) float64 bool


## [База]


1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?


In [70]:
print(f"Позиция самого дорого заказа: {np.argmax(amount)}")
print(f"Сумма: {amount[np.argmax(amount)]}")
print(f"Клиент: {cust[np.argmax(amount)]}")
print(f"Город: {city[np.argmax(amount)]}")

Позиция самого дорого заказа: 1001
Сумма: 9999990.0
Клиент: TEST
Город: Алматы


2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.


In [71]:
print(f"Сумма оплаченных заказов: {np.sum(amount[paid])}")
print(f"Сумма неоплаченных заказов: {np.sum(amount[~paid])}")

Сумма оплаченных заказов: 101909360.0
Сумма неоплаченных заказов: 7974000.0


3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [72]:
print(f"Заказы из Шымкента: {len(city[city == 'Шымкент'])}")
print(f"Доля от всех заказов: {len(city[city == 'Шымкент']) / len(city) * 100:.2f}%")

Заказы из Шымкента: 804
Доля от всех заказов: 14.72%


## [Стандарт]


4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?

In [73]:
no_d = amount[(paid) & (amount > 50000)]
d = amount[(paid) & (amount > 50000)] * 0.85
print(f"Магазин недополучит со всеми заказами: {np.sum(no_d - d)}") 

test = "TEST"
no_dt = amount[(paid) & (amount > 50000) & (cust != test)]
no_t = amount[(paid) & (amount > 50000) & (cust != test)]  * 0.85
print(f"Магазин недополучит со всеми заказами и без клиента TEST: {np.sum(no_dt - no_t)}")

Магазин недополучит со всеми заказами: 5702422.5
Магазин недополучит со всеми заказами и без клиента TEST: 1202427.0


Тестовый клиент TEST искажает расчёты: его крупный заказ значительно увеличивает сумму потерь из-за применения скидки. Для объективной оценки финансового эффекта тестовые данные нужно исключить.

5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.


In [74]:
every_city = {}
unique_cities = np.unique(city)
for c in unique_cities:
    every_city[c] = len(days[(city == c) & (1 <= days) &  (days <= 2)]) / len(days[city == c]) * 100
for c, p in every_city.items():
    print(f"{c}: {p:.2f}%")

Актобе: 62.22%
Алматы: 64.53%
Астана: 63.53%
Атырау: 64.38%
Караганда: 67.84%
Шымкент: 29.85%


6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [75]:
avg = amount[paid].mean()
print(f"Средний чек: {avg:.2f}")

avg = amount[(paid) & (cust != test)].mean()
print(f"Средний чек без TEST: {avg:.2f}")

Средний чек: 20780.86
Средний чек без TEST: 14672.39


## [Челлендж]


7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?


In [76]:
from time import perf_counter
start = perf_counter()
summ = 0
for p, a in zip(paid, amount):
    if p:
        summ += a
print(f"Сумма: {summ}")
end = perf_counter()
print(f"Скорость при for: {end - start}")

start_np = perf_counter()
np.sum(amount[paid])
end_np = perf_counter()
print(f"Скорость при numpy: {end_np - start_np}")

Сумма: 101909360.0
Скорость при for: 0.0008677499999976135
Скорость при numpy: 0.00015183300001808675


numpy выполнил операцию в 3.37 раза быстрее, так-как использует язык C

8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [ ]:
np.random.seed(42)
print(f"Количество покупок в группе A: {np.sum(np.random.rand(1000) < 0.05)}")
print(f"Количество покупок в группе B: {np.sum(np.random.rand(1000) < 0.06)}")


rnd = np.random.rand(100, 1000)
rnd2 = np.random.rand(100, 1000)
test_a = np.sum(rnd < 0.05, axis=1)
test_b = np.sum(rnd2 < 0.06, axis=1)
print(f"B оказался лучше при: {np.sum(test_a < test_b)} случаях")


Количество покупок в группе A: 54
Количество покупок в группе B: 58
B оказался лучше при: 84 случаях


Несмотря на разницу в вероятности покупки всего в 1 п.п., группа B показала лучший результат в 84 из 100 экспериментов. При выборке в 1000 клиентов на группу случайные колебания всё ещё могут привести к тому, что B окажется хуже A. Чем больше выборка, тем меньше влияние случайности и тем точнее оценка разницы между группами.